# 08 — Final full-training fit and locked-test evaluation

Stage 08 runs **no cross-validation**. The five-fold scaffold-grouped CV metrics of each model's best Optuna trial are read from Stage 07. Each of ExtraTrees, CatBoost and SVR is fitted once on the complete training set, using the same loader, row order and ordered 250-feature list as Stage 07, with **every** Optuna best parameter applied and audited against the fitted estimator. Each fitted model is then scored once on the locked scaffold test set.

Reported on the locked test set: RMSE, MSE, MAE, median AE, R² (= Q²F2), Q²F1, Q²F3, Pearson, Spearman, Kendall, CCC, bias, calibration slope/intercept, Golbraikh–Tropsha criteria, fraction within 2-fold / 0.5 / 10-fold, 95% scaffold-cluster bootstrap CIs, paired RMSE differences, CV-to-test gaps, and error by similarity to the training set. Baselines: training mean, Morgan Tanimoto 1-NN, similarity-weighted 5-NN. An equal-weight ensemble is shown as supplementary only.

**Interpretation:** the primary model is the one with the lowest Stage 07 CV RMSE, never chosen from test results. The 250 features were selected on the full training partition before CV, and parameters were selected by Optuna, so CV scores may be optimistic. The locked test set is the independent external estimate and must not be used to retune. No imputation is used.

In [1]:
import sys
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Open this notebook from the BRD4 QSAR project folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import OUTPUTS
SELECTED_FEATURES_FILE = OUTPUTS / "05_feature_selection/tables/selected_features.csv"
if not SELECTED_FEATURES_FILE.is_file():
    raise FileNotFoundError(f"Missing Stage 05 fixed-feature list: {SELECTED_FEATURES_FILE}")
selected_features = pd.read_csv(SELECTED_FEATURES_FILE)["feature"].astype(str).tolist()
if len(selected_features) != 250 or len(set(selected_features)) != 250:
    raise ValueError(f"Expected 250 unique Stage 05 features; found {len(selected_features)} entries and {len(set(selected_features))} unique names.")
print("Project root:", PROJECT_ROOT)
print(f"Validated fixed Stage 05 features: {len(selected_features)} unique features")
print("Stage 08 will use this exact ordered list for each full-training final fit; Stage 07 already used it for CV.")


Project root: c:\Users\USER\Documents\E\Code
Validated fixed Stage 05 features: 250 unique features
Stage 08 will use this exact ordered list for each full-training final fit; Stage 07 already used it for CV.


In [2]:
from src.final_evaluation import run
# on_invalid_test='error' stops if any locked-test molecule has a missing/non-finite selected feature.
# Use run(on_invalid_test='exclude') to score the remaining molecules and log exclusions.
results = run()
results

Fitted ExtraTreesRegressor: 6 Optuna parameters applied and verified.
Fitted CatBoostRegressor: 5 Optuna parameters applied and verified.
Fitted SVR: 4 Optuna parameters applied and verified.

Locked-test results (CV columns are Stage 07 values; no CV was run here):
                                model  cv_rank  cv_mean_fold_rmse  test_rmse  test_rmse_ci_low  test_rmse_ci_high  test_mae  test_r2  test_pearson  test_spearman  test_ccc  test_frac_within_2fold  gap_rmse_test_minus_cv
                                  SVR      1.0             0.5886     0.5621            0.5188             0.6046    0.3954   0.8111        0.9009         0.9003    0.8982                  0.5331                 -0.0266
                  ExtraTreesRegressor      2.0             0.5900     0.5771            0.5241             0.6264    0.4009   0.8008        0.8952         0.8958    0.8879                  0.5339                 -0.0129
                    CatBoostRegressor      3.0             0.5999     0.5

,comparison_type,model,parameter_source,parameters,cv_mean_fold_rmse,cv_sd_fold_rmse,cv_mean_fold_mse,cv_sd_fold_mse,cv_mean_fold_mae,cv_sd_fold_mae,...,test_r2_ci_low,test_r2_ci_high,test_pearson_ci_low,test_pearson_ci_high,test_spearman_ci_low,test_spearman_ci_high,test_ccc_ci_low,test_ccc_ci_high,n_bootstrap,spearman_max_similarity_vs_abs_error
0,QSAR model,SVR,Optuna best parameters,"{""C"": 18.684324183616088, ""epsilon"": 0.0020663...",0.588636,0.041992,0.347903,0.049049,0.415455,0.028367,...,0.762277,0.846888,8.748204e-01,9.208299e-01,0.868663,0.918768,8.722439e-01,9.180614e-01,2000,-0.211417
1,QSAR model,ExtraTreesRegressor,Optuna best parameters,"{""bootstrap"": true, ""max_depth"": 40, ""max_feat...",0.589988,0.041758,0.349481,0.048724,0.424642,0.036030,...,0.749262,0.838452,8.665420e-01,9.166351e-01,0.862343,0.915877,8.586529e-01,9.097701e-01,2000,-0.196713
2,QSAR model,CatBoostRegressor,Optuna best parameters,"{""depth"": 8, ""iterations"": 1000, ""l2_leaf_reg""...",0.599910,0.042042,0.361307,0.049826,0.436760,0.035780,...,0.748858,0.834619,8.657856e-01,9.148773e-01,0.860925,0.912264,8.553215e-01,9.058745e-01,2000,-0.146360
3,Supplementary ensemble,Equal-weight ensemble (supplementary),Mean of the three fitted models,{},NaN,NaN,NaN,NaN,NaN,NaN,...,0.770327,0.851726,8.780250e-01,9.238356e-01,0.873048,0.922213,8.695804e-01,9.170697e-01,2000,-0.175737
4,Baseline,Morgan Tanimoto weighted 5-NN,Training data only,"{""neighbors"": 5, ""weighting"": ""Tanimoto simila...",NaN,NaN,NaN,NaN,NaN,NaN,...,0.731697,0.832031,8.652728e-01,9.152445e-01,0.855911,0.909420,8.590678e-01,9.118440e-01,2000,-0.191572
5,Baseline,Morgan Tanimoto 1-NN,Training data only,"{""neighbors"": 1, ""weighting"": ""none""}",NaN,NaN,NaN,NaN,NaN,NaN,...,0.609486,0.772106,8.141218e-01,8.889528e-01,0.811077,0.881763,8.078175e-01,8.872644e-01,2000,-0.176746
6,Baseline,Training mean,Training data only,"{""statistic"": ""mean of training pIC50""}",NaN,NaN,NaN,NaN,NaN,NaN,...,-0.063760,-0.000013,-1.063188e-15,6.690223e-16,NaN,NaN,-1.838496e-30,1.210433e-30,2000,0.150957


In [3]:
import pandas as pd
from src.utils import OUTPUTS
T = OUTPUTS / '08_final_model_evaluation/tables'
display(pd.read_csv(T / 'final_model_comparison.csv').round(4))
display(pd.read_csv(T / 'paired_rmse_differences_bootstrap.csv').round(4))
display(pd.read_csv(T / 'applicability_domain_error.csv').round(4))
display(pd.read_csv(T / 'final_model_parameters.csv').query("origin == 'Optuna best parameter'"))

,comparison_type,model,cv_rank,is_primary_model,cv_mean_fold_rmse,cv_sd_fold_rmse,cv_mean_fold_mae,cv_mean_fold_r2,test_n,test_rmse,...,test_pearson,test_spearman,test_spearman_ci_low,test_spearman_ci_high,test_ccc,test_ccc_ci_low,test_ccc_ci_high,gap_rmse_test_minus_cv,gap_rmse_in_cv_sd,gap_r2_test_minus_cv
0,QSAR model,SVR,1.0,True,0.5886,0.0420,0.4155,0.7541,1283,0.5621,...,0.9009,0.9003,0.8687,0.9188,0.8982,0.8722,0.9181,-0.0266,-0.6324,0.0570
1,QSAR model,ExtraTreesRegressor,2.0,False,0.5900,0.0418,0.4246,0.7516,1283,0.5771,...,0.8952,0.8958,0.8623,0.9159,0.8879,0.8587,0.9098,-0.0129,-0.3091,0.0492
2,QSAR model,CatBoostRegressor,3.0,False,0.5999,0.0420,0.4368,0.7432,1283,0.5810,...,0.8939,0.8935,0.8609,0.9123,0.8842,0.8553,0.9059,-0.0189,-0.4489,0.0549
3,Supplementary ensemble,Equal-weight ensemble (supplementary),NaN,False,NaN,NaN,NaN,NaN,1283,0.5531,...,0.9042,0.9043,0.8730,0.9222,0.8971,0.8696,0.9171,NaN,NaN,NaN
4,Baseline,Morgan Tanimoto weighted 5-NN,NaN,False,NaN,NaN,NaN,NaN,1283,0.5948,...,0.8930,0.8894,0.8559,0.9094,0.8886,0.8591,0.9118,NaN,NaN,NaN
5,Baseline,Morgan Tanimoto 1-NN,NaN,False,NaN,NaN,NaN,NaN,1283,0.7114,...,0.8533,0.8502,0.8111,0.8818,0.8506,0.8078,0.8873,NaN,NaN,NaN
6,Baseline,Training mean,NaN,False,NaN,NaN,NaN,NaN,1283,1.2940,...,NaN,NaN,NaN,NaN,-0.0000,-0.0000,0.0000,NaN,NaN,NaN


,model_a,model_b,rmse_a_minus_rmse_b,ci_low,ci_high,prob_a_lower_rmse,ci_excludes_zero
0,ExtraTreesRegressor,CatBoostRegressor,-0.0040,-0.0207,0.0168,0.6490,False
1,ExtraTreesRegressor,SVR,0.0150,-0.0133,0.0440,0.1535,False
2,ExtraTreesRegressor,Equal-weight ensemble (supplementary),0.0239,0.0115,0.0393,0.0005,True
3,ExtraTreesRegressor,Training mean,-0.7170,-0.8335,-0.6007,1.0000,True
4,ExtraTreesRegressor,Morgan Tanimoto 1-NN,-0.1343,-0.2031,-0.0564,1.0000,True
5,ExtraTreesRegressor,Morgan Tanimoto weighted 5-NN,-0.0178,-0.0549,0.0214,0.8110,False
6,CatBoostRegressor,SVR,0.0190,-0.0081,0.0453,0.0930,False
7,CatBoostRegressor,Equal-weight ensemble (supplementary),0.0279,0.0156,0.0397,0.0000,True
8,CatBoostRegressor,Training mean,-0.7130,-0.8213,-0.6057,1.0000,True
9,CatBoostRegressor,Morgan Tanimoto 1-NN,-0.1304,-0.1920,-0.0585,1.0000,True


,model,similarity_bin,n,rmse,mae,bias_obs_minus_pred
0,ExtraTreesRegressor,0.0-0.3,5,1.3017,1.0970,-0.6041
1,ExtraTreesRegressor,0.3-0.4,10,0.7674,0.6451,-0.5541
2,ExtraTreesRegressor,0.4-0.5,29,1.0347,0.7772,-0.0570
3,ExtraTreesRegressor,0.5-0.6,84,0.7823,0.5826,-0.0450
4,ExtraTreesRegressor,0.6-0.7,263,0.6087,0.4353,-0.0088
5,ExtraTreesRegressor,0.7-1.0,892,0.5111,0.3548,-0.0154
6,CatBoostRegressor,0.0-0.3,5,1.3114,1.1192,-0.5788
7,CatBoostRegressor,0.3-0.4,10,0.7035,0.5550,-0.4465
8,CatBoostRegressor,0.4-0.5,29,1.0479,0.7622,-0.0478
9,CatBoostRegressor,0.5-0.6,84,0.7478,0.5600,-0.0148


,model,parameter,value,origin
0,ExtraTreesRegressor,bootstrap,true,Optuna best parameter
3,ExtraTreesRegressor,max_depth,40,Optuna best parameter
4,ExtraTreesRegressor,max_features,0.46298618941691183,Optuna best parameter
8,ExtraTreesRegressor,min_samples_leaf,1,Optuna best parameter
9,ExtraTreesRegressor,min_samples_split,2,Optuna best parameter
12,ExtraTreesRegressor,n_estimators,900,Optuna best parameter
27,CatBoostRegressor,depth,8,Optuna best parameter
33,CatBoostRegressor,iterations,1000,Optuna best parameter
34,CatBoostRegressor,l2_leaf_reg,0.10998433083295822,Optuna best parameter
38,CatBoostRegressor,learning_rate,0.047204334288835526,Optuna best parameter
